## **Importing the dependencies**

In [1]:
print("Nikhil, you are ready to go !")

Nikhil, you are ready to go !


In [2]:
# import os
# import numpy as np
# import nibabel as nib
# import torch
# from torch.utils.data import Dataset, DataLoader
# from torchvision import transforms
# from PIL import Image

# class LungCancerDataset(Dataset):
#     def __init__(self, root_dir):
#         self.image_paths = []
#         self.mask_paths = []
#         patient_dirs = sorted(os.listdir(root_dir))
#         for patient in patient_dirs:
#             data_dir = os.path.join(root_dir, patient, 'data')
#             mask_dir = os.path.join(root_dir, patient, 'masks')
            
#             for file_name in sorted(os.listdir(data_dir)):
#                 img_path = os.path.join(data_dir, file_name)
#                 mask_path = os.path.join(mask_dir, file_name)  # same name
#                 self.image_paths.append(img_path)
#                 self.mask_paths.append(mask_path)

#     def __len__(self):
#         return len(self.image_paths)

#     def __getitem__(self, idx):
#         image = np.load(self.image_paths[idx])
#         mask = np.load(self.mask_paths[idx])
#         image = torch.from_numpy(image).unsqueeze(0).float()
#         mask = torch.from_numpy(mask).unsqueeze(0).float()
        
#         return image, mask


# # Custom Dataset for Pneumonia (COVID) Dataset (2D images)
# class PneumoniaDataset(Dataset):
#     def __init__(self, ct_scan_dir, mask_dir, transform=None):
#         self.ct_scan_dir = ct_scan_dir
#         self.mask_dir = mask_dir
#         self.transform = transform
        
#         self.ct_scan_paths = [os.path.join(ct_scan_dir, file) for file in os.listdir(ct_scan_dir)]
#         self.mask_paths = [os.path.join(mask_dir, file) for file in os.listdir(mask_dir)]

#     def __len__(self):
#         return len(self.ct_scan_paths)

#     def __getitem__(self, idx):
#         # Load .nii files
#         ct_scan_nii = nib.load(self.ct_scan_paths[idx])
#         mask_nii = nib.load(self.mask_paths[idx])
        
#         # Extract 2D slices and normalize them
#         ct_scan = ct_scan_nii.get_fdata()
#         mask = mask_nii.get_fdata()
        
#         # Normalize the CT scan pixel values
#         ct_scan = ct_scan / np.max(ct_scan)
        
#         # Take a slice (let's assume the middle slice for simplicity)
#         slice_idx = ct_scan.shape[2] // 2
#         ct_scan_slice = ct_scan[:, :, slice_idx]
#         mask_slice = mask[:, :, slice_idx]
        
#         # Convert to Image format (2D, shape 256x256)
#         ct_scan_slice = Image.fromarray(ct_scan_slice)
#         ct_scan_slice = ct_scan_slice.resize((256, 256))
#         mask_slice = Image.fromarray(mask_slice)
#         mask_slice = mask_slice.resize((256, 256))
#         mask_slice = np.array(mask_slice)
#         mask_slice = np.where(mask_slice > 0, 2, 0)
        
#         # Apply transformations if available
#         if self.transform:
#             ct_scan_slice = self.transform(ct_scan_slice)
#             mask_slice = self.transform(mask_slice)

#         return ct_scan_slice, mask_slice

# # Combine both datasets into one
# class CombinedDataset(Dataset):
#     def __init__(self, lung_cancer_data_dir, pneumonia_ct_scan_dir, pneumonia_mask_dir, transform=None):
#         self.lung_cancer_dataset = LungCancerDataset(lung_cancer_data_dir)
#         self.pneumonia_dataset = PneumoniaDataset(pneumonia_ct_scan_dir, pneumonia_mask_dir, transform)
        
#     def __len__(self):
#         return len(self.lung_cancer_dataset) + len(self.pneumonia_dataset)
    
#     def __getitem__(self, idx):
#         if idx < len(self.lung_cancer_dataset):
#             return self.lung_cancer_dataset[idx]
#         else:
#             return self.pneumonia_dataset[idx - len(self.lung_cancer_dataset)]

# # Define transformations (optional)
# transform = transforms.Compose([
#     transforms.ToTensor(),
# ])

# # Define dataset directories
# lung_cancer_train_dir = "/kaggle/input/lung-cancer-segment/train"
# lung_cancer_val_dir = "/kaggle/input/lung-cancer-segment/val"
# pneumonia_ct_scan_dir = "/kaggle/input/covid19-ct-scans/ct_scans"
# pneumonia_mask_dir = "/kaggle/input/covid19-ct-scans/infection_mask"

# # Create Combined Dataset and DataLoader
# dataset = CombinedDataset(lung_cancer_train_dir, pneumonia_ct_scan_dir, pneumonia_mask_dir, transform)
# dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

# # Now, you can loop through the DataLoader for your model training
# for images, masks in dataloader:
#     # Your training code here
#     print(images.shape, masks.shape)  # This will print (batch_size, channels, 256, 256)
#     break

In [3]:
from warnings import filterwarnings

filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
import os
import gc
import random
from IPython.display import display
import ipywidgets as widgets
from ipywidgets import interact

from tqdm.auto import tqdm
import torch.nn.functional as F
from torchvision.transforms.v2 import GaussianNoise
from torchmetrics import JaccardIndex, Precision, Recall, Specificity, F1Score, AUROC
import torch.optim as optim

from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr_skimage
from skimage.metrics import mean_squared_error as mse_skimage
from skimage.metrics import hausdorff_distance
from scipy.ndimage import distance_transform_edt
from tensorflow.keras.preprocessing.image import load_img
from keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset
from torch.utils.data import DataLoader


import torch

def Dice(preds, targets, smooth=1e-6, threshold=None):
    
    if preds.shape != targets.shape:
        raise ValueError("Predictions and targets must have the same shape.")
    
    # Apply thresholding for binary or multi-class case
    if threshold is not None:
        preds = (preds > threshold).float()

    # Flatten tensors except for batch and channel dimensions
    preds = preds.flatten(2)  # (B, C, H*W)
    targets = targets.flatten(2)

    intersection = (preds * targets).sum(dim=-1)
    union = preds.sum(dim=-1) + targets.sum(dim=-1)

    dice = (2.0 * intersection + smooth) / (union + smooth)
    return dice.mean()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

2025-06-25 16:37:39.765551: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750869459.950666      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750869460.006917      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [4]:
# import os
# import glob
# import nibabel as nib
# import numpy as np
# from torch.utils.data import Dataset, DataLoader
# from sklearn.model_selection import train_test_split
# import torch
# import torch.nn.functional as F

# class LungCT2DSliceDataset(Dataset):
#     def __init__(self, image_paths, label_paths, transform=None):
#         self.slices = []  # List of (image_slice, label_slice) pairs
#         self.transform = transform

#         for img_path, lbl_path in zip(image_paths, label_paths):
#             img_nii = nib.load(img_path).get_fdata().astype(np.float32)
#             lbl_nii = nib.load(lbl_path).get_fdata().astype(np.int64)

#             # Ensure same shape
#             if img_nii.shape != lbl_nii.shape:
#                 continue

#             # Loop over depth slices (assume (H, W, D))
#             for i in range(img_nii.shape[2]):
#                 img_slice = img_nii[:, :, i]
#                 lbl_slice = lbl_nii[:, :, i]

#                 # Optional: ignore empty masks
#                 if np.sum(lbl_slice) == 0:
#                     continue

#                 self.slices.append((img_slice, lbl_slice))

#     def __len__(self):
#         return len(self.slices)

#     def __getitem__(self, idx):
#         img, mask = self.slices[idx]

#         # Normalize image to [0, 1]
#         img = (img - img.min()) / (img.max() - img.min() + 1e-8)

#         # Convert to torch tensor & add channel dimension
#         img = torch.from_numpy(img).unsqueeze(0)  # shape: (1, H, W)
#         mask = torch.from_numpy(mask).unsqueeze(0).float()  # shape: (1, H, W)

#         # Resize to 56x56 using bilinear for img, nearest for mask
#         img = F.interpolate(img.unsqueeze(0), size=(256, 256), mode='bilinear', align_corners=False).squeeze(0)
#         mask = F.interpolate(mask.unsqueeze(0), size=(256, 256), mode='nearest').squeeze(0)

#         return img, mask
# def get_2d_slice_dataloaders(root_dir, test_size=0.1, val_size=0.1, batch_size=8):
#     image_dir = os.path.join(root_dir, 'imagesTr')
#     label_dir = os.path.join(root_dir, 'labelsTr')

#     image_paths = sorted(glob.glob(os.path.join(image_dir, "*2.nii*"))) + sorted(glob.glob(os.path.join(image_dir, "*3.nii*")))
#     label_paths = sorted(glob.glob(os.path.join(label_dir, "*2.nii*"))) + sorted(glob.glob(os.path.join(label_dir, "*3.nii*")))

#     # Match paths by filename
#     image_paths_base = [os.path.basename(p) for p in image_paths]
#     label_paths_dict = {os.path.basename(p): p for p in label_paths}
#     paired_imgs, paired_lbls = [], []

#     for img in image_paths_base:
#         if img in label_paths_dict:
#             paired_imgs.append(os.path.join(image_dir, img))
#             paired_lbls.append(label_paths_dict[img])

#     # Split
#     X_temp, X_test, y_temp, y_test = train_test_split(paired_imgs, paired_lbls, test_size=test_size, random_state=42)
#     val_ratio = val_size / (1 - test_size)
#     X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=val_ratio, random_state=42)

#     # Datasets
#     train_dataset = LungCT2DSliceDataset(X_train, y_train)
#     val_dataset = LungCT2DSliceDataset(X_val, y_val)
#     test_dataset = LungCT2DSliceDataset(X_test, y_test)

#     # Loaders
#     train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
#     val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=2)
#     test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=2)

#     return train_loader, val_loader, test_loader
# root = "/kaggle/input/medical-segmentation-decathlon-lung"
# train_loader, val_loader, test_loader = get_2d_slice_dataloaders(root, batch_size=8)

# for images, masks in train_loader:
#     print("Image batch shape:", images.shape)  # (8, 1, 56, 56)
#     print("Mask batch shape: ", masks.shape)   # (8, 1, 56, 56)
#     break


In [5]:
import os
import numpy as np
from torch.utils.data import Dataset, DataLoader
import torch
import torchvision.transforms as transforms
from PIL import Image

class LungSegmentationDataset(Dataset):
    def __init__(self, root_dir):
        self.image_paths = []
        self.mask_paths = []
        patient_dirs = sorted(os.listdir(root_dir))
        for patient in patient_dirs:
            data_dir = os.path.join(root_dir, patient, 'data')
            mask_dir = os.path.join(root_dir, patient, 'masks')
            
            for file_name in sorted(os.listdir(data_dir)):
                img_path = os.path.join(data_dir, file_name)
                mask_path = os.path.join(mask_dir, file_name)  # same name
                self.image_paths.append(img_path)
                self.mask_paths.append(mask_path)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = np.load(self.image_paths[idx])
        mask = np.load(self.mask_paths[idx])
        image = torch.from_numpy(image).unsqueeze(0).float()
        mask = torch.from_numpy(mask).unsqueeze(0).float()
        
        return image, mask

train_dataset = LungSegmentationDataset(root_dir='/kaggle/input/lung-cancer-segment/train')

val_dataset = LungSegmentationDataset(root_dir='/kaggle/input/lung-cancer-segment/val')

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=4)


In [6]:
len(train_loader), len(val_loader)

(1804, 167)

In [7]:
for i,j in val_loader:
    print(i.shape, j.shape)
    break

torch.Size([8, 1, 256, 256]) torch.Size([8, 1, 256, 256])


## **Importing Libraries for UNet**

In [8]:
import random
from tqdm import tqdm
import csv
import time 

import torch
import torch.nn as nn
from torchvision import models
from torch.nn.functional import relu
import torch.nn.functional as F

## **UNet Architecture**

In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class UNet1(nn.Module):
    def __init__(self, n_class=1):
        super().__init__()

        self.e11 = nn.Conv2d(1, 64, kernel_size=3, padding=1)
        self.e12 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.e21 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.e22 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.e31 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.e32 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.e41 = nn.Conv2d(256, 512, kernel_size=3, padding=1)
        self.e42 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.e51 = nn.Conv2d(512, 1024, kernel_size=3, padding=1)
        self.e52 = nn.Conv2d(1024, 1024, kernel_size=3, padding=1)

        self.upconv1 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.d11 = nn.Conv2d(1024, 512, kernel_size=3, padding=1)
        self.d12 = nn.Conv2d(512, 512, kernel_size=3, padding=1)

        self.upconv2 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.d21 = nn.Conv2d(512, 256, kernel_size=3, padding=1)
        self.d22 = nn.Conv2d(256, 256, kernel_size=3, padding=1)

        self.upconv3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.d31 = nn.Conv2d(256, 128, kernel_size=3, padding=1)
        self.d32 = nn.Conv2d(128, 128, kernel_size=3, padding=1)

        self.upconv4 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.d41 = nn.Conv2d(128, 64, kernel_size=3, padding=1)
        self.d42 = nn.Conv2d(64, 64, kernel_size=3, padding=1)

        self.outconv = nn.Conv2d(64, n_class, kernel_size=1)    # For Segmentation mask
        self.finalconv = nn.Conv2d(64, 1, kernel_size=1)   # For Denoised image

    def forward(self, x):
        x = F.relu(self.e11(x))
        x1 = F.relu(self.e12(x))
        x = self.pool1(x1)

        x = F.relu(self.e21(x))
        x2 = F.relu(self.e22(x))
        x = self.pool2(x2)

        x = F.relu(self.e31(x))
        x3 = F.relu(self.e32(x))
        x = self.pool3(x3)

        x = F.relu(self.e41(x))
        x4 = F.relu(self.e42(x))
        x = self.pool4(x4)

        x = F.relu(self.e51(x))
        x = F.relu(self.e52(x))

        x = self.upconv1(x)
        x = torch.cat([x, x4], dim=1)
        x = F.relu(self.d11(x))
        x = F.relu(self.d12(x))

        x = self.upconv2(x)
        x = torch.cat([x, x3], dim=1)
        x = F.relu(self.d21(x))
        x = F.relu(self.d22(x))

        x = self.upconv3(x)
        x = torch.cat([x, x2], dim=1)
        x = F.relu(self.d31(x))
        x = F.relu(self.d32(x))

        x = self.upconv4(x)
        x = torch.cat([x, x1], dim=1)
        x = F.relu(self.d41(x))
        x = F.relu(self.d42(x))

        out = self.outconv(x) 
        return out


# YOLO insipired Arch for Multi Task Segmetation

### Utils

In [10]:


def fuse_conv(conv, norm):
    fused_conv = torch.nn.Conv2d(conv.in_channels,
                                 conv.out_channels,
                                 kernel_size=conv.kernel_size,
                                 stride=conv.stride,
                                 padding=conv.padding,
                                 groups=conv.groups,
                                 bias=True).requires_grad_(False).to(conv.weight.device)

    w_conv = conv.weight.clone().view(conv.out_channels, -1)
    w_norm = torch.diag(norm.weight.div(torch.sqrt(norm.eps + norm.running_var)))
    fused_conv.weight.copy_(torch.mm(w_norm, w_conv).view(fused_conv.weight.size()))

    b_conv = torch.zeros(conv.weight.size(0), device=conv.weight.device) if conv.bias is None else conv.bias
    b_norm = norm.bias - norm.weight.mul(norm.running_mean).div(torch.sqrt(norm.running_var + norm.eps))
    fused_conv.bias.copy_(torch.mm(w_norm, b_conv.reshape(-1, 1)).reshape(-1) + b_norm)

    return fused_conv


class Conv(torch.nn.Module):
    def __init__(self, in_ch, out_ch, activation, k=1, s=1, p=0, g=1):
        super().__init__()
        self.conv = torch.nn.Conv2d(in_ch, out_ch, k, s, p, groups=g, bias=False)
        self.norm = torch.nn.BatchNorm2d(out_ch, eps=0.001, momentum=0.03)
        self.relu = activation

    def forward(self, x):
        return self.relu(self.norm(self.conv(x)))

    def fuse_forward(self, x):
        return self.relu(self.conv(x))


class Residual(torch.nn.Module):
    def __init__(self, ch, e=0.5):
        super().__init__()
        self.conv1 = Conv(ch, int(ch * e), torch.nn.SiLU(), k=3, p=1)
        self.conv2 = Conv(int(ch * e), ch, torch.nn.SiLU(), k=3, p=1)

    def forward(self, x):
        return x + self.conv2(self.conv1(x))


class C3K(torch.nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv1 = Conv(in_ch, out_ch // 2, torch.nn.SiLU())
        self.conv2 = Conv(in_ch, out_ch // 2, torch.nn.SiLU())
        self.conv3 = Conv(2 * (out_ch // 2), out_ch, torch.nn.SiLU())
        self.res_m = torch.nn.Sequential(Residual(out_ch // 2, e=1.0),
                                         Residual(out_ch // 2, e=1.0))

    def forward(self, x):
        y = self.res_m(self.conv1(x))
        return self.conv3(torch.cat((y, self.conv2(x)), dim=1))


class C3K2(torch.nn.Module):
    def __init__(self, in_ch, out_ch, n, csp, r):
        super().__init__()
        self.conv1 = Conv(in_ch, 2 * (out_ch // r), torch.nn.SiLU())
        self.conv2 = Conv((2 + n) * (out_ch // r), out_ch, torch.nn.SiLU())

        if not csp:
            self.res_m = torch.nn.ModuleList(Residual(out_ch // r) for _ in range(n))
        else:
            self.res_m = torch.nn.ModuleList(C3K(out_ch // r, out_ch // r) for _ in range(n))

    def forward(self, x):
        y = list(self.conv1(x).chunk(2, 1))
        y.extend(m(y[-1]) for m in self.res_m)
        return self.conv2(torch.cat(y, dim=1))


class SPP(torch.nn.Module):
    def __init__(self, in_ch, out_ch, k=5):
        super().__init__()
        self.conv1 = Conv(in_ch, in_ch // 2, torch.nn.SiLU())
        self.conv2 = Conv(in_ch * 2, out_ch, torch.nn.SiLU())
        self.res_m = torch.nn.MaxPool2d(k, stride=1, padding=k // 2)

    def forward(self, x):
        x = self.conv1(x)
        y1 = self.res_m(x)
        y2 = self.res_m(y1)
        return self.conv2(torch.cat(tensors=[x, y1, y2, self.res_m(y2)], dim=1))


class Attention(torch.nn.Module):

    def __init__(self, ch, num_head):
        super().__init__()
        self.num_head = num_head
        self.dim_head = ch // num_head
        self.dim_key = self.dim_head // 2
        self.scale = self.dim_key ** -0.5

        self.qkv = Conv(ch, ch + self.dim_key * num_head * 2, torch.nn.Identity())

        self.conv1 = Conv(ch, ch, torch.nn.Identity(), k=3, p=1, g=ch)
        self.conv2 = Conv(ch, ch, torch.nn.Identity())

    def forward(self, x):
        b, c, h, w = x.shape

        qkv = self.qkv(x)
        qkv = qkv.view(b, self.num_head, self.dim_key * 2 + self.dim_head, h * w)

        q, k, v = qkv.split([self.dim_key, self.dim_key, self.dim_head], dim=2)

        attn = (q.transpose(-2, -1) @ k) * self.scale
        attn = attn.softmax(dim=-1)

        x = (v @ attn.transpose(-2, -1)).view(b, c, h, w) + self.conv1(v.reshape(b, c, h, w))
        return self.conv2(x)


class PSABlock(torch.nn.Module):

    def __init__(self, ch, num_head):
        super().__init__()
        self.conv1 = Attention(ch, num_head)
        self.conv2 = torch.nn.Sequential(Conv(ch, ch * 2, torch.nn.SiLU()),
                                         Conv(ch * 2, ch, torch.nn.Identity()))

    def forward(self, x):
        x = x + self.conv1(x)
        return x + self.conv2(x)


class PSA(torch.nn.Module):
    def __init__(self, ch, n):
        super().__init__()
        self.conv1 = Conv(ch, 2 * (ch // 2), torch.nn.SiLU())
        self.conv2 = Conv(2 * (ch // 2), ch, torch.nn.SiLU())
        self.res_m = torch.nn.Sequential(*(PSABlock(ch // 2, ch // 128) for _ in range(n)))

    def forward(self, x):
        x, y = self.conv1(x).chunk(2, 1)
        return self.conv2(torch.cat(tensors=(x, self.res_m(y)), dim=1))


class DarkNet(torch.nn.Module):
    def __init__(self, width, depth, csp):
        super().__init__()
        self.p1 = []
        self.p2 = []
        self.p3 = []
        self.p4 = []
        self.p5 = []

        # p1/2
        self.p1.append(Conv(width[0], width[1], torch.nn.SiLU(), k=3, s=2, p=1))
        # p2/4
        self.p2.append(Conv(width[1], width[2], torch.nn.SiLU(), k=3, s=2, p=1))
        self.p2.append(C3K2(width[2], width[3], depth[0], csp[0], r=4))
        # p3/8
        self.p3.append(Conv(width[3], width[3], torch.nn.SiLU(), k=3, s=2, p=1))
        self.p3.append(C3K2(width[3], width[4], depth[1], csp[0], r=4))
        # p4/16
        self.p4.append(Conv(width[4], width[4], torch.nn.SiLU(), k=3, s=2, p=1))
        self.p4.append(C3K2(width[4], width[4], depth[2], csp[1], r=2))
        # p5/32
        self.p5.append(Conv(width[4], width[5], torch.nn.SiLU(), k=3, s=2, p=1))
        self.p5.append(C3K2(width[5], width[5], depth[3], csp[1], r=2))
        self.p5.append(SPP(width[5], width[5]))
        self.p5.append(PSA(width[5], depth[4]))

        self.p1 = torch.nn.Sequential(*self.p1)
        self.p2 = torch.nn.Sequential(*self.p2)
        self.p3 = torch.nn.Sequential(*self.p3)
        self.p4 = torch.nn.Sequential(*self.p4)
        self.p5 = torch.nn.Sequential(*self.p5)

    def forward(self, x):
        p1 = self.p1(x)
        p2 = self.p2(p1)
        p3 = self.p3(p2)
        p4 = self.p4(p3)
        p5 = self.p5(p4)
        return p1, p2, p3, p4, p5

class DarkFPN(torch.nn.Module):
    def __init__(self, width, depth, csp):
        super().__init__()
        self.up = torch.nn.Upsample(scale_factor=2)
        self.h1 = C3K2(width[4] + width[5], width[4], depth[5], csp[0], r=2)
        self.h2 = C3K2(width[4] + width[4], width[3], depth[5], csp[0], r=2)
        self.h3 = Conv(width[3], width[3], torch.nn.SiLU(), k=3, s=2, p=1)
        self.h4 = C3K2(width[3] + width[4], width[4], depth[5], csp[0], r=2)
        self.h5 = Conv(width[4], width[4], torch.nn.SiLU(), k=3, s=2, p=1)
        self.h6 = C3K2(width[4] + width[5], width[5], depth[5], csp[1], r=2)

    def forward(self, x):
        p1, p2, p3, p4, p5 = x
        p4 = self.h1(torch.cat([self.up(p5), p4], dim=1))
        p3 = self.h2(torch.cat([self.up(p4), p3], dim=1))
        p4 = self.h4(torch.cat([self.h3(p3), p4], dim=1))
        p5 = self.h6(torch.cat([self.h5(p4), p5], dim=1))
        return p1, p2, p3, p4, p5


## Un-Optimized Model

In [11]:


class SegmentationHead(torch.nn.Module):
    def __init__(self, p5_channels, p4_channels, p3_channels, p2_channels, p1_channels, num_classes):
        super().__init__()
        self.up1 = torch.nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv1 = Conv(p5_channels + p4_channels, p5_channels, torch.nn.SiLU(), k=3, p=1)

        self.up2 = torch.nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv2 = Conv(p5_channels + p3_channels, p5_channels // 2, torch.nn.SiLU(), k=3, p=1)

        self.up3 = torch.nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv3 = Conv(p5_channels // 2 + p2_channels, p5_channels // 4, torch.nn.SiLU(), k=3, p=1)

        self.up4 = torch.nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv4 = Conv(p5_channels // 4 + p1_channels, p5_channels // 8, torch.nn.SiLU(), k=3, p=1)

        self.up5 = torch.nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv5 = Conv(p5_channels // 8, p5_channels // 8, torch.nn.SiLU(), k=3, p=1)

        self.final = torch.nn.Conv2d(p5_channels // 8, num_classes, kernel_size=1)

    def forward(self, p5, p4, p3, p2, p1):
        x = self.up1(p5)  # 16 → 32
        x = torch.cat([x, p4], dim=1)
        x = self.conv1(x)

        x = self.up2(x)  # 32 → 64
        x = torch.cat([x, p3], dim=1)
        x = self.conv2(x)

        x = self.up3(x)  # 64 → 128
        x = torch.cat([x, p2], dim=1)
        x = self.conv3(x)

        x = self.up4(x)  # 128 → 256
        x = torch.cat([x, p1], dim=1)
        x = self.conv4(x)

        x = self.up5(x)  # 256 → 512
        x = self.conv5(x)

        return self.final(x)

class DenoisingHead(torch.nn.Module):
    def __init__(self, p5_channels, p4_channels, p3_channels, p2_channels, p1_channels):
        super().__init__()
        self.up1 = torch.nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv1 = Conv(p5_channels + p4_channels, p5_channels, torch.nn.SiLU(), k=3, p=1)

        self.up2 = torch.nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv2 = Conv(p5_channels + p3_channels, p5_channels // 2, torch.nn.SiLU(), k=3, p=1)

        self.up3 = torch.nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv3 = Conv(p5_channels // 2 + p2_channels, p5_channels // 4, torch.nn.SiLU(), k=3, p=1)

        self.up4 = torch.nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv4 = Conv(p5_channels // 4 + p1_channels, p5_channels // 8, torch.nn.SiLU(), k=3, p=1)

        self.up5 = torch.nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv5 = Conv(p5_channels // 8, p5_channels // 8, torch.nn.SiLU(), k=3, p=1)

        self.final = torch.nn.Conv2d(p5_channels // 8, 1, kernel_size=1)

    def forward(self, p5, p4, p3, p2, p1):
        x = self.up1(p5)
        x = torch.cat([x, p4], dim=1)
        x = self.conv1(x)

        x = self.up2(x)
        x = torch.cat([x, p3], dim=1)
        x = self.conv2(x)

        x = self.up3(x)
        x = torch.cat([x, p2], dim=1)
        x = self.conv3(x)

        x = self.up4(x)
        x = torch.cat([x, p1], dim=1)
        x = self.conv4(x)

        x = self.up5(x)
        x = self.conv5(x)

        return torch.sigmoid(self.final(x))


class MT_UNet_YOLO_Large(torch.nn.Module):
    def __init__(self, width, depth, csp, num_classes):
        super().__init__()
        self.net = DarkNet(width, depth, csp)
        self.fpn = DarkFPN(width, depth, csp)

        # Dummy input to infer FPN output shapes
        img_dummy = torch.zeros(1, width[0], 512, 512)
        p1, p2, p3, p4, p5 = self.fpn(self.net(img_dummy))  # Get actual shapes

        # Task heads now use correct channel counts from FPN outputs
        self.seg_head = SegmentationHead(p5_channels=p5.shape[1], p4_channels=p4.shape[1],p3_channels=p3.shape[1],p2_channels=p2.shape[1],p1_channels=p1.shape[1], num_classes=num_classes)

    def forward(self, x):
        p1, p2, p3, p4, p5 = self.fpn(self.net(x))

        seg_out = self.seg_head(p5, p4, p3, p2, p1)

        return seg_out

    def fuse(self):
        for m in self.modules():
            if type(m) is Conv and hasattr(m, 'norm'):
                m.conv = fuse_conv(m.conv, m.norm)
                m.forward = m.fuse_forward
                delattr(m, 'norm')
        return self




# ab = MT_UNet_YOLO_Large([1, 32, 64, 128, 256, 512], [1, 1, 1, 1, 1, 1], [False, True], 1)
# nik= ab(torch.randn(1,1,512,512))


## Training

# UNet Segmentation

In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchmetrics import StructuralSimilarityIndexMeasure, MultiScaleStructuralSimilarityIndexMeasure, PeakSignalNoiseRatio
import numpy as np
import pandas as pd
from tqdm import tqdm  # Add this import at the top

import os
from datetime import datetime
import time

# -------------------------------
# 📊 Metric Functions
# -------------------------------

def dice_score(preds, targets, threshold=0.5, smooth=1e-6):
    preds = (preds > threshold).float()
    intersection = (preds * targets).sum()
    total = preds.sum() + targets.sum()
    return (2.0 * intersection + smooth) / (total + smooth)

def precision(preds, targets, threshold=0.5, smooth=1e-6):
    preds = (preds > threshold).float()
    tp = (preds * targets).sum()
    fp = preds.sum() - tp
    return (tp + smooth) / (tp + fp + smooth)

def recall(preds, targets, threshold=0.5, smooth=1e-6):
    preds = (preds > threshold).float()
    tp = (preds * targets).sum()
    fn = targets.sum() - tp
    return (tp + smooth) / (tp + fn + smooth)

def specificity(preds, targets, threshold=0.5, smooth=1e-6):
    preds = (preds > threshold).float()
    tn = ((1 - preds) * (1 - targets)).sum()
    fp = preds.sum() - (preds * targets).sum()
    return (tn + smooth) / (tn + fp + smooth)

def f1_score(precision, recall, beta=1.0, smooth=1e-6):
    return (1 + beta**2) * (precision * recall + smooth) / (beta**2 * precision + recall + smooth)

def rmse(preds, targets):
    return torch.sqrt(torch.mean((preds - targets) ** 2))

def snr(preds, targets):
    signal = targets.mean()
    noise = (targets - preds).abs().mean()
    return signal / noise

# -------------------------------
# 📦 Metrics Handler
# -------------------------------

class MetricsHandler:
    def __init__(self, device='cuda'):
        self.ssim = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)
        self.msssim = MultiScaleStructuralSimilarityIndexMeasure(data_range=1.0).to(device)
        self.psnr = PeakSignalNoiseRatio(data_range=1.0).to(device)

    def compute(self, pred_denoise, clean_img):
        ssim = self.ssim(pred_denoise, clean_img).item()
        msssim = self.msssim(pred_denoise, clean_img).item()
        psnr = self.psnr(pred_denoise, clean_img).item()
        return ssim, msssim, psnr

# -------------------------------
# 🧠 Training Loop
# -------------------------------

def seg_train_model(model, train_loader, val_loader, device, name, num_epochs=50):
    save_dir = name
    os.makedirs(save_dir, exist_ok=True)

    # Losses
    bce_loss = nn.BCEWithLogitsLoss()
    mse_loss = nn.MSELoss()

    # Optimizer
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    # Metrics
    metric_handler = MetricsHandler(device=device)

    # Log DataFrame with custom column order
    columns = [
        'Epoch', 'Total Loss', 'Dice Score', 'Time (s)',
        'Precision', 'Recall', 'F1 Score', 'Specificity', 
        'Val Total Loss',  'Val Dice Score',  'Val Time (s)', 
        'Val Precision', 'Val Recall', 'Val F1 Score', 'Val Specificity',

    ]
    log_df = pd.DataFrame(columns=columns)

    best_val_loss = float('inf')

    for epoch in range(num_epochs):
        metrics = {col: 0.0 for col in columns}  # Initialize metrics dict

        for phase in ['train', 'val']:
            dataloader = train_loader if phase == 'train' else val_loader
            model.train() if phase == 'train' else model.eval()

            batch_metrics = {
                'total_loss': [],
                'dice': [], 'precision': [], 'recall': [], 'specificity': [],
                
            }

            start_time = time.time()

            with torch.set_grad_enabled(phase == 'train'):
                for noisy_img, mask in tqdm(dataloader, desc=f"{phase.capitalize()} Epoch {epoch+1}", leave=False):
                    noisy_img = noisy_img.to(device)
                    mask = mask.to(device)

                    seg_mask_logits = model(noisy_img)

                    # Losses
                    loss_seg = bce_loss(seg_mask_logits, mask)
                    total_loss = loss_seg

                    if phase == 'train':
                        optimizer.zero_grad()
                        total_loss.backward()
                        optimizer.step()

                    # Segmentation metrics
                    seg_probs = torch.sigmoid(seg_mask_logits)
                    dice = dice_score(seg_probs, mask).item()
                    prec = precision(seg_probs, mask).item()
                    rec = recall(seg_probs, mask).item()
                    spec = specificity(seg_probs, mask).item()
                    f1 = f1_score(prec, rec)

                    # Append batch metrics
                    batch_metrics['total_loss'].append(total_loss.item())
                    batch_metrics['dice'].append(dice)
                    batch_metrics['precision'].append(prec)
                    batch_metrics['recall'].append(rec)
                    batch_metrics['specificity'].append(spec)
                  

            # Aggregate metrics
            end_time = time.time()
            avg = {k: np.mean(v) for k, v in batch_metrics.items()}
            time_taken = end_time - start_time

            # Update metrics dict
            prefix = '' if phase == 'train' else 'Val '
            metrics[f'{prefix}Total Loss'] = avg['total_loss']
            metrics[f'{prefix}Dice Score'] = avg['dice']
            metrics[f'{prefix}Time (s)'] = time_taken
            metrics[f'{prefix}Precision'] = avg['precision']
            metrics[f'{prefix}Recall'] = avg['recall']
            metrics[f'{prefix}F1 Score'] = f1_score(avg['precision'], avg['recall'])
            metrics[f'{prefix}Specificity'] = avg['specificity']
      

        # Append to log
        log_df = pd.concat([log_df, pd.DataFrame([metrics])], ignore_index=True)

        # Save best model
        if metrics['Val Total Loss'] < best_val_loss:
            best_val_loss = metrics['Val Total Loss']
            best_model_path = os.path.join(save_dir, f'{name}_best_val_loss.pt')
            torch.save(model.state_dict(), best_model_path)

        # Print progress
        print(f"Epoch {epoch+1}/{num_epochs} | "
              f"Train Loss: {metrics['Total Loss']:.4f} | "
              f"Train Dice: {metrics['Dice Score']:.4f} | "
              f"Val Loss: {metrics['Val Total Loss']:.4f} | "
              f"Val Dice: {metrics['Val Dice Score']:.4f} | "
              )

    # Save logs
    log_path = os.path.join(save_dir, f'{name}_Training.csv')
    log_df.to_csv(log_path, index=False, float_format='%.4f')
    print(f"Training complete. Logs saved to {log_path}")

    return model



In [13]:
def seg_test_model(model, test_loader, device, name):
    save_dir = name
    os.makedirs(save_dir, exist_ok=True)

    # Loss functions
    bce_loss = nn.BCEWithLogitsLoss()
    mse_loss = nn.MSELoss()

    # Metrics handler
    metric_handler = MetricsHandler(device=device)

    # Column order
    columns = ['Total Loss','Dice Score', 'Time (s)',
        'Precision', 'Recall', 'F1 Score', 'Specificity', 
        
    ]

    # Initialize metrics dictionary
    metrics = {col: 0.0 for col in columns}

    # Batch metrics
    batch_metrics = {'total_loss': [],'dice': [], 'precision': [], 'recall': [], 'specificity': []}

    model.eval()  # Set model to evaluation mode
    start_time = time.time()

    with torch.no_grad():
        for noisy_img, mask in tqdm(test_loader, desc=f"TEST- Epoch ", leave=False):
            
            noisy_img = noisy_img.to(device)
            mask = mask.to(device)

            seg_mask_logits = model(noisy_img)

            # Losses
            loss_seg = bce_loss(seg_mask_logits, mask)
            total_loss = loss_seg

            # Segmentation metrics
            seg_probs = torch.sigmoid(seg_mask_logits)
            dice = dice_score(seg_probs, mask).item()
            prec = precision(seg_probs, mask).item()
            rec = recall(seg_probs, mask).item()
            spec = specificity(seg_probs, mask).item()
            f1 = f1_score(prec, rec)

            # Append batch metrics
            batch_metrics['total_loss'].append(total_loss.item())
            batch_metrics['dice'].append(dice)
            batch_metrics['precision'].append(prec)
            batch_metrics['recall'].append(rec)
            batch_metrics['specificity'].append(spec)


    # Aggregate metrics
    avg = {k: np.mean(v) for k, v in batch_metrics.items()}
    time_taken = time.time() - start_time

    # Fill metrics dictionary (test = validation phase)
    metrics.update({
        'Dice Score': avg['dice'],
        'Time (s)': time_taken,
        'Precision': avg['precision'],
        'Recall': avg['recall'],
        'F1 Score': f1_score(avg['precision'], avg['recall']),
        'Specificity': avg['specificity'],
  

        
    })

    # Create DataFrame and save
    log_df = pd.DataFrame([metrics])
    log_path = os.path.join(save_dir, f'{name}_Testing.csv')
    log_df.to_csv(log_path, index=False, float_format='%.4f')

    print(f"Test complete. Results saved to {log_path}")
    print(f"Test Dice: {avg['dice']:.4f}")

    return log_df



In [14]:
model = UNet1(n_class=1).to(device)
model.load_state_dict(torch.load('/kaggle/input/unet-mdcancer/pytorch/default/1/U-Net/U-Net_best_val_loss.pt'))

<All keys matched successfully>

In [15]:
test_results = seg_test_model(model, val_loader, device, name='UNet')

Test complete. Results saved to UNet/UNet_Testing.csv
Test Dice: 0.8481


In [16]:
depth = [1,1,1,1,1,1]
NUM_CLASS = 1
NUM_EPOCHS = 30
multi_tasks_model_dict = {
'U-Net': UNet1(n_class=NUM_CLASS),
'YOU-Net': MT_UNet_YOLO_Large([1, 64, 128, 256, 512, 512], depth, [True, True], NUM_CLASS)
}

for name, model in multi_tasks_model_dict.items():
    model = model.to(device)
    trained_model = seg_train_model(model,train_loader=train_loader,val_loader=val_loader,
                                device=device, name = name, num_epochs=NUM_EPOCHS)
    test_results = seg_test_model(trained_model, val_loader, device, name=name)
    print(f"---------------------------{name} training and testing is completed--------------------------------------")

Epoch 1/30 | Train Loss: 0.0132 | Train Dice: 0.4119 | Val Loss: 0.0015 | Val Dice: 0.8922 | 


Epoch 2/30 | Train Loss: 0.0046 | Train Dice: 0.4202 | Val Loss: 0.0014 | Val Dice: 0.8922 | 


Epoch 3/30 | Train Loss: 0.0040 | Train Dice: 0.4113 | Val Loss: 0.0015 | Val Dice: 0.8922 | 


Epoch 4/30 | Train Loss: 0.0024 | Train Dice: 0.4436 | Val Loss: 0.0012 | Val Dice: 0.8872 | 


Epoch 5/30 | Train Loss: 0.0012 | Train Dice: 0.5977 | Val Loss: 0.0009 | Val Dice: 0.8769 | 


...

Epoch 27/30 | Train Loss: 0.0002 | Train Dice: 0.9195 | Val Loss: 0.0014 | Val Dice: 0.8766 | 


Epoch 28/30 | Train Loss: 0.0001 | Train Dice: 0.9288 | Val Loss: 0.0013 | Val Dice: 0.8616 | 


Epoch 29/30 | Train Loss: 0.0001 | Train Dice: 0.9286 | Val Loss: 0.0013 | Val Dice: 0.8394 | 


Epoch 30/30 | Train Loss: 0.0001 | Train Dice: 0.9276 | Val Loss: 0.0016 | Val Dice: 0.8489 | 
Training complete. Logs saved to YOU-Net/YOU-Net_Training.csv


Test complete. Results saved to YOU-Net/YOU-Net_Testing.csv
Test Dice: 0.8489
---------------------------YOU-Net training and testing is completed--------------------------------------


In [17]:
StoptheTraining

NameError: name 'StoptheTraining' is not defined

In [ ]:
# test_model = MT_UNet_YOLO_Large([1, 64, 128, 256, 512, 512], [1,1,1,1,1,1], [True, True], 1).to(device)
# test_model.load_state_dict(torch.load('/kaggle/input/you-net-lung-cancer/pytorch/default/1/YOU-Net_best_val_loss.pt'))

In [ ]:
import glob

path ='/kaggle/input/lung-cancer-segment'
val_data_path = os.path.join(path, 'val', '57', 'data')
val_mask_path = os.path.join(path, 'val', '57', 'masks')
image_files = sorted(glob.glob(os.path.join(val_data_path, "*.npy")))
mask_files = sorted(glob.glob(os.path.join(val_mask_path, "*.npy")))
all_images = [np.load(f) for f in image_files]
all_masks = [np.load(f) for f in mask_files]

image_volume = np.stack(all_images, axis=0) # Stack along a new axis (depth)
mask_volume = np.stack(all_masks, axis=0)   # Stack along a new axis (depth)

predicted_masks_list = []
model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
with torch.no_grad():
    for i in tqdm(range(image_volume.shape[0]), desc="Processing slices"):
        img_slice = image_volume[i, :, :]
        img_tensor = torch.from_numpy(img_slice).unsqueeze(0).unsqueeze(0).float().to(device) # Add batch and channel
        output = model(img_tensor)
        predicted_mask_slice = torch.sigmoid(output).squeeze().cpu().numpy() > 0.5 # Remove batch and channel, move to cpu, convert to numpy, threshold
        predicted_masks_list.append(predicted_mask_slice)

predicted_mask_volume = np.stack(predicted_masks_list, axis=0)

In [ ]:
imgTarget = image_volume.transpose(1,2,0)
imgMask = mask_volume.transpose(1,2,0)
predImg = predicted_mask_volume.transpose(1,2,0)

In [ ]:
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from skimage import measure


In [ ]:
vertices, faces, _, _ = measure.marching_cubes(predImg, level=0.5)
ctvertices, ctfaces, _, _ = measure.marching_cubes(imgTarget, level=0.5)
maskvertices, maskfaces, _, _ = measure.marching_cubes(imgMask, level=0.5)

# Optional: Display the number of vertices and faces for debugging
print(f"Inferred Mask: {len(vertices)} vertices, {len(faces)} faces")
print(f"Input Scan: {len(ctvertices)} vertices, {len(ctfaces)} faces")
print(f"Annotated Mask: {len(maskvertices)} vertices, {len(maskfaces)} faces")

In [ ]:
import os
import nibabel as nib
import numpy as np
import torch
import torch.nn.functional as F

# Define paths
dataInputPath = '/kaggle/input/medical-segmentation-decathlon-lung'
imagePathInput = os.path.join(dataInputPath, 'imagesTr/')
maskPathInput = os.path.join(dataInputPath, 'labelsTr/')

targetImageFile = 'lung_003.nii'
targetMaskFile = 'lung_003.nii'

targetImagePath = os.path.join(imagePathInput, targetImageFile)
targetMaskPath = os.path.join(maskPathInput, targetMaskFile)

def preprocess_image_and_mask(img_path, mask_path):
    # Load NIfTI files
    img_nii = nib.load(img_path).get_fdata().astype(np.float32)
    lbl_nii = nib.load(mask_path).get_fdata().astype(np.int64)

    # Ensure shape match
    if img_nii.shape != lbl_nii.shape:
        raise ValueError("Image and mask shapes do not match.")

    images_list = []
    mask_list= []

    image_data = np.zeros((256, 256, img_nii.shape[2]))

    mask_data= np.zeros((256, 256, img_nii.shape[2]))

    for i in range(img_nii.shape[2]):
        img_slice = img_nii[:, :, i]
        lbl_slice = lbl_nii[:, :, i]


        # Normalize image to [0,1]
        img_slice = (img_slice - img_slice.min()) / (img_slice.max() - img_slice.min() + 1e-8)

        # Convert to tensor and add channel dim
        img_tensor = torch.from_numpy(img_slice).unsqueeze(0)  # (1, H, W)
        mask_tensor = torch.from_numpy(lbl_slice).unsqueeze(0).float()  # (1, H, W)

        # Resize to (256, 256)
        img_tensor = F.interpolate(img_tensor.unsqueeze(0), size=(256, 256), mode='bilinear', align_corners=False).squeeze(0)
        mask_tensor = F.interpolate(mask_tensor.unsqueeze(0), size=(256, 256), mode='nearest').squeeze(0)
        images_list.append(img_tensor)
        mask_list.append(mask_tensor)
        image_data[:,:,i] = img_tensor
        mask_data[:,:,i] = mask_tensor
        # all_slices.append((img_tensor, mask_tensor))

    return image_data, mask_data

# Run preprocessing
imgTarget, imgMask = preprocess_image_and_mask(targetImagePath, targetMaskPath)


In [ ]:
predictions = np.zeros((256, 256, 288))

with torch.no_grad():
    for idx in range(imgTarget.shape[-1]):
        img1 = imgTarget[:, :, idx][np.newaxis, np.newaxis, :, :] 
        img_tensor = torch.from_numpy(img1).float().cuda()
        out = test_model(img_tensor)
        prediction_slice = out.squeeze().cpu() 
        predictions[:, :, idx] = (torch.sigmoid(prediction_slice).numpy()>0.5).astype(np.int64)

In [ ]:
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
# from stl import mesh
from skimage import measure

predImg = predictions

# Generate vertices and faces using marching cubes
vertices, faces, _, _ = measure.marching_cubes(predImg, level=0.5)
ctvertices, ctfaces, _, _ = measure.marching_cubes(imgTarget, level=0.5)
maskvertices, maskfaces, _, _ = measure.marching_cubes(imgMask, level=0.5)

# Optional: Display the number of vertices and faces for debugging
print(f"Inferred Mask: {len(vertices)} vertices, {len(faces)} faces")
print(f"Input Scan: {len(ctvertices)} vertices, {len(ctfaces)} faces")
print(f"Annotated Mask: {len(maskvertices)} vertices, {len(maskfaces)} faces")

In [ ]:
# prompt: calculate the dice score of predImg and imgMask

def calculate_dice_score(predImg, imgMask):
    """
    Calculates the Dice Similarity Coefficient (DSC) between two binary masks.

    Args:
        predImg (np.ndarray): Predicted binary mask.
        imgMask (np.ndarray): Ground truth binary mask.

    Returns:
        float: Dice score.
    """
    # Ensure both inputs are binary (0 or 1)
    predImg_binary = (predImg > 0.5).astype(np.float32)
    imgMask_binary = (imgMask > 0.5).astype(np.float32)

    intersection = np.sum(predImg_binary * imgMask_binary)
    sum_masks = np.sum(predImg_binary) + np.sum(imgMask_binary)

    # Avoid division by zero
    if sum_masks == 0:
        return 1.0  # Or 0.0, depending on how you define Dice for empty masks

    dice = (2. * intersection) / sum_masks
    return dice

# Calculate and print the Dice score
dice_score = calculate_dice_score(predImg, imgMask)
print(f"Dice Score: {dice_score}")

In [ ]:
!pip install numpy-stl -q

In [ ]:
from stl import mesh

def dataToMesh(vert, faces):
    stl_mesh = mesh.Mesh(np.zeros(faces.shape[0], dtype=mesh.Mesh.dtype))
    for i, f in enumerate(faces):
        for j in range(3):
            stl_mesh.vectors[i][j] = vert[f[j], :]
    return stl_mesh

# Convert and save all .stl files
output_path = './'  # Define your output directory
inference_mesh = dataToMesh(vertices, faces)
inference_mesh.save(output_path + 'Inferenced_lung.stl')

# input_mesh = dataToMesh(ctvertices, ctfaces)
# input_mesh.save(output_path + 'Input_lung_003.stl')

mask_mesh = dataToMesh(maskvertices, maskfaces)
mask_mesh.save(output_path + 'Mask_lung.stl')

print("STL files saved successfully.")

In [ ]:
import open3d as o3d

stl_file_path = "/content/Inferenced_lung_003.stl"  
mesh = o3d.io.read_triangle_mesh(stl_file_path)
print("_")
pointcloud = mesh.sample_points_poisson_disk(100000)
xyz_load = np.asarray(pointcloud.points, dtype=np.float32)
print('xyz_load shape', xyz_load.shape)